# **Proyek ETL Pipeline & Analisis Pemasaran Perbankan (Bank Marketing)**
## **Layer 2: Silver Layer (Data Cleansing, Deduplication & Feature Engineering)**

Notebook ini merupakan tahap kedua dari arsitektur Medallion. Pada tahap ini dilakukan:
1. **Pemuatan Data dari Bronze Layer:** Mengambil seluruh data mentah yang di-ingest secara batch dari `lakehouse/bronze/bank_marketing`.
2. **Deduplikasi & Pembersihan Kualitas Data:** Menghapus baris ganda (termasuk duplikasi akibat siklus *auto-reset* Bronze) untuk memastikan hanya data unik yang masuk ke tahap analitik.
3. **Feature Engineering & Enrichment:** Menambahkan atribut bisnis kategorisasi `AgeGroup` (`Muda (<30)`, `Paruh Baya (30-50)`, `Tua (>50)`).
4. **Penyimpanan ke Silver Layer:** Menyimpan dataset yang bersih ke Parquet (`lakehouse/silver/bank_marketing`) serta ekspor ke CSV tunggal.

---
### **1. Inisialisasi Sesi Spark**

In [ ]:
# Instal pyspark jika dijalankan di Google Colab terpisah
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

# Inisialisasi Spark Session
spark = SparkSession.builder \
    .appName("BankMarketing_02_Silver") \
    .getOrCreate()

print("Spark Session untuk Layer Silver berhasil diinisialisasi.")

---
### **2. Membaca Data dari Layer Bronze (Incremental Batches)**

Memuat data Parquet yang dihasilkan dari ingestion batch di Layer Bronze.

In [ ]:
bronze_path = "lakehouse/bronze/bank_marketing"

# Membaca data dari Layer Bronze
bronze_data = spark.read.parquet(bronze_path)
row_before = bronze_data.count()

print(f"Data Bronze dimuat dari: {bronze_path}")
print(f"Jumlah baris awal dari Bronze (gabungan seluruh batch): {row_before}")
bronze_data.printSchema()
bronze_data.show(5)

---
### **3. Data Cleansing & Deduplikasi Data**

Karena Layer Bronze beroperasi secara *append-only* (dan dapat mengalami *auto-reset* per siklus), **Layer Silver bertanggung jawab melakukan deduplikasi** berdasarkan kolom-kolom bisnis asli nasabah.

In [ ]:
# Daftar kolom fitur asli (tanpa metadata ingest)
feature_cols = [c for c in bronze_data.columns if c not in ["batch_id", "cycle_id", "ingested_at", "source_file"]]

# Deduplikasi berdasarkan fitur asli
silver_clean = bronze_data.dropDuplicates(subset=feature_cols)

row_clean = silver_clean.count()
print(f"Jumlah baris sebelum pembersihan : {row_before}")
print(f"Jumlah baris unik setelah deduplikasi : {row_clean}")
print(f"Total baris duplikat yang dibersihkan  : {row_before - row_clean}")

---
### **4. Feature Engineering: Kategorisasi Kelompok Usia (`AgeGroup`)**

Menambahkan atribut baru yang membagi usia klien ke dalam 3 segmen:
* `Muda (<30)` : Usia di bawah 30 tahun.
* `Paruh Baya (30-50)` : Usia antara 30 hingga 50 tahun.
* `Tua (>50)` : Usia di atas 50 tahun.

In [ ]:
# Menambahkan Kolom AgeGroup
silver_final = silver_clean.withColumn(
    "AgeGroup",
    when(col("age") < 30, "Muda (<30)")
    .when((col("age") >= 30) & (col("age") <= 50), "Paruh Baya (30-50)")
    .otherwise("Tua (>50)")
)

print("Skema Data Silver setelah Feature Engineering:")
silver_final.printSchema()

# Distribusi data per kelompok usia
print("\nDistribusi Kelompok Usia (AgeGroup):")
silver_final.groupBy("AgeGroup").count().show()

---
### **5. Penyimpanan ke Silver Layer & Ekspor CSV Tunggal**

In [ ]:
# Simpan ke Silver Layer (format Parquet)
silver_path = "lakehouse/silver/bank_marketing"
silver_final.write.mode("overwrite").parquet(silver_path)

row_after = silver_final.count()
print(f"Layer Silver Selesai.")
print(f"Lokasi penyimpanan: {silver_path}")
print(f"Jumlah baris tersimpan: {row_after}")

# Verifikasi membaca kembali data Silver
print("\n=== Sampel Data di Layer Silver ===")
spark.read.parquet(silver_path).select("age", "AgeGroup", "job", "balance", "duration", "y").show(5)

# Ekspor ke CSV Tunggal untuk downstream consumption
silver_export_path = "lakehouse/exports/silver_export"
silver_final.coalesce(1).write.mode("overwrite").option("header", "true").csv(silver_export_path)
print(f"Ekspor Silver ke CSV tunggal berhasil di: {silver_export_path}")

---
### **Ringkasan Layer Silver**
* Data mentah dari berbagai batch Bronze telah dibersihkan dan dideduplikasi secara konsisten.
* Fitur segmentasi `AgeGroup` berhasil dibuat.
* Data siap dikonsumsi oleh **Layer Gold** (`03_Gold_Layer.ipynb`) untuk pembuatan agregasi data mart bisnis.